In [1]:
%cd ../
%ls

/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit
CH_All_Countries/ dist/             idr/              venv/
Makefile          edges_all.json    nodes_all.json    wag_toolkit/
README.md         environment.yml   pyproject.toml
careers/          geographies/      requirements.txt


In [2]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [3]:
grants_ref = pd.read_excel('geographies/CH_files/Manual Tagging of Grants with C&H goals.xlsx', sheet_name='Total C&H Awards_SM tagging')
grants_ref = grants_ref[grants_ref['C&H/C&H in partnership/wider portfolio (new)'].isin(['C&H', 'C&H in partnership'])]
grants_list = list(set(grants_ref['Grant Reference'].to_list()))

In [4]:
# Only Wellcome
candh_intersection = pd.read_excel('geographies/CH_files/C&HPublicationExtraction.xlsx', sheet_name='Combined Awards')


# All C&H
# candh_intersection = pd.read_parquet('geographies/CH_files/climate_health_intersection_just_ids.parquet')

In [5]:
# Select relevant Grants for C&H
candh_intersection = candh_intersection[candh_intersection['Wellcome Grant Reference'].isin(grants_list)]

pub_ids = list(set(candh_intersection['Publication ID'].tolist()))

In [6]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""
loc = Locations(dummy_query)

query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication)
            WHERE p.dimensions_publication_id IN {}
            RETURN p.dimensions_publication_id AS dimensions_publication_id, p.year AS year,
                a.institutions AS grid_id"""
loc.lookup_query(query=query, lookup=pub_ids)

100%|██████████| 2/2 [00:00<00:00,  3.08it/s]


In [7]:
data =pd.DataFrame(loc.data)
data = data.explode("grid_id").dropna()
data.head()

,dimensions_publication_id,year,grid_id
8,pub.1146763677,2022.0,grid.189504.1
9,pub.1146763677,2022.0,grid.8991.9
10,pub.1145226527,2022.0,grid.8991.9
11,pub.1145226527,2022.0,['grid.16463.36']
12,pub.1145226527,2022.0,grid.16463.36


In [8]:
loc._clean_grid_ids()
loc.extract_edges()
loc.extract_locations("country")

loc.convert_edges()
loc.calculate_adjacency_matrices()

100%|██████████| 1/1 [00:00<00:00,  7.75it/s]
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep cu

2022.0
2021.0
2023.0
2017.0
2019.0
2015.0
2018.0
2016.0
2020.0


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.


In [9]:
import numpy as np
loc.adjacency_matrices = {np.int64(key): value for key, value in loc.adjacency_matrices.items()}

In [10]:
countries_class = pd.read_excel('geographies/CH_files/CLASS.xlsx', sheet_name='List of economies')
lmic_list = countries_class[countries_class['Income group'].isin(['Low income', 'Lower middle income', 'Upper middle income'])]
lmic_list.head()

,Economy,Code,Region,Income group,Lending category
0,Afghanistan,AFG,South Asia,Low income,IDA
1,Albania,ALB,Europe & Central Asia,Upper middle income,IBRD
2,Algeria,DZA,Middle East & North Africa,Upper middle income,IBRD
5,Angola,AGO,Sub-Saharan Africa,Lower middle income,IBRD
7,Argentina,ARG,Latin America & Caribbean,Upper middle income,IBRD


In [11]:
lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
lmic_list['Economy'] = lmic_list['Economy'].replace("Iran, Islamic Rep.", "Iran")

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_33927/708080782.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_33927/708080782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_33927/708080782.py:3: SettingWithCopyW

In [12]:
# Only account for LMIC
only_lmic = True

if only_lmic:
    for year in loc.adjacency_matrices:
        for country in loc.adjacency_matrices[year]['All'].index:
            if country == 'All' or country == 'total':
                continue
            for country2 in loc.adjacency_matrices[year]['All'][country].index:
                if country in list(lmic_list['Economy']) or country2 in list(lmic_list['Economy']):
                    continue
                else:
                    loc.adjacency_matrices[year]['All'][country][country2]=0

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_33927/191586841.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  loc.adjacency_matrices[year]['All'][country][country2]=0
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykern

In [13]:
loc.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.1, directed=False, threshold=0, font = {"size": 20, "face": "Helvetica Neue"}, node_count='total')

In [14]:
dirname = 'CH_LMIC_Countries'
loc.to_visjs(vis_name="locations", directed=False, template='geographies/locations.html', dirname=dirname)
loc._to_json("nodes_all.json", loc.vis_nodes)
loc._to_json("edges_all.json", loc.vis_edges)

In [15]:
import os
import shutil

with(open(f'{dirname}/edges.json','r')) as f:
    edges = json.load(f)

os.rename(f'{dirname}/nodes.json', f'{dirname}/nodes_all.json')
shutil.copyfile('geographies/positions.json', f'{dirname}/positions.json')

def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict


with(open(f'{dirname}/dict_edges_all.json','w')) as f:
        json.dump(edges_to_dict(edges), f)